In [2]:
# ─────────────────────────────────────────────
# Part 1. 라이브러리 설치 및 드라이브 마운트
# ─────────────────────────────────────────────
# 공식 faiss-gpu 대신 최신 환경과 호환되는 faiss-gpu-cu12를 설치합니다.
!pip install sentence-transformers langchain-huggingface langchain-community faiss-gpu-cu12

import json
import numpy as np
from google.colab import drive
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 구글 드라이브 마운트 (팝업창에서 권한 허용 필요)
drive.mount('/content/drive')

print("라이브러리 임포트 및 드라이브 마운트 완료")

  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency re

In [5]:
# ─────────────────────────────────────────────
# Part 2. 청크 파일 로드
# ─────────────────────────────────────────────

FILE_PATH = "/content/ingredient_chunks_preset1.json"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"총 청크 수: {len(all_chunks)}")
print(f"\n[샘플 확인]")
print(f"page_content: {all_chunks[0]['page_content']}")
print(f"metadata: {all_chunks[0]['metadata']}")

총 청크 수: 25855

[샘플 확인]
page_content: [(20S)-프로토파낙사다이올] 기능: AI Anti-aging/Wrinkle Care AI Antioxidant 식 피부보호제, 피부컨디셔닝제(유연제), 피부컨디셔닝제(보습제)
metadata: {'ingredient_ko': '(20S)-프로토파낙사다이올', 'ingredient_en': '(20S)-Protopanaxadiol', 'coos_score': 0, 'hw_ewg': None, 'pc_rating': 0, 'chunk_type': 'basic_info', 'chunk_weight': 0.33}


In [6]:
# ─────────────────────────────────────────────
# Part 3. LangChain Document 객체로 변환
# ─────────────────────────────────────────────
docs = [
    Document(
        page_content=chunk["page_content"],
        metadata=chunk["metadata"]
    )
    for chunk in all_chunks
]

print(f"Document 변환 완료: {len(docs)}개")

Document 변환 완료: 25855개


In [7]:
# ─────────────────────────────────────────────
# Part 4. 임베딩 모델 로드 (GPU 할당)
# ─────────────────────────────────────────────
# device를 "cuda"로 설정하여 T4 GPU를 사용하도록 합니다.

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cuda"},    # ★ 핵심: GPU 사용
    encode_kwargs={"normalize_embeddings": True},
)

print("임베딩 모델 로드 완료 (GPU)")

# 테스트 (768차원 확인)
test_vector = embedding_model.embed_query("나이아신아마이드 EWG 등급")
print(f"벡터 차원 수: {len(test_vector)}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 모델 로드 완료 (GPU)
벡터 차원 수: 768


In [9]:
# ─────────────────────────────────────────────
# Part 5. FAISS 인덱스 구축
# ─────────────────────────────────────────────
# GPU를 사용하므로 로컬 CPU 대비 훨씬 빠르게 진행됩니다.

print("FAISS 인덱스 구축 시작... (GPU 가동 중 🚀)")

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model,
)

print("FAISS 인덱스 구축 완료!")
print(f"저장된 벡터 수: {vectorstore.index.ntotal}")

FAISS 인덱스 구축 시작... (GPU 가동 중 🚀)
FAISS 인덱스 구축 완료!
저장된 벡터 수: 25855


In [10]:
# ─────────────────────────────────────────────
# Part 6. FAISS 인덱스 로컬(드라이브) 저장
# ─────────────────────────────────────────────
# 생성된 인덱스를 구글 드라이브에 저장하여 나중에 바로 불러올 수 있게 합니다.

FAISS_SAVE_PATH = "/content/drive/MyDrive/data/faiss_index_preset1"

vectorstore.save_local(FAISS_SAVE_PATH)
print(f"FAISS 인덱스 저장 완료: {FAISS_SAVE_PATH}")

FAISS 인덱스 저장 완료: /content/drive/MyDrive/data/faiss_index_preset1
